In [ ]:
# CELL 1 — Install only if needed
# Uncomment in a fresh environment.
#
# %pip install -U numpy scipy pandas matplotlib pillow tqdm torch transformers

In [ ]:
# CELL 2 — Imports and reproducibility

import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import jensenshannon
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_rel
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# CELL 3 — Paths / configuration
#
# Replace these paths with the actual paths used by the project.

PROJECT_ROOT = Path(".").resolve()

ANNOTATED_CSV = PROJECT_ROOT / "annotated_all_models.csv"
IMAGE_DIR = PROJECT_ROOT / "images" / "val"
SALIENCY_DIR = PROJECT_ROOT / "maps" / "val"

OUTPUT_DIR = PROJECT_ROOT / "results"
CACHE_DIR = OUTPUT_DIR / "cache"
FIGURES_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Salesforce/blip-image-captioning-base"

# Start with a small run for debugging. Set to None for the full dataset
# after all checks pass.
MAX_IMAGES = 50

print("Annotation file:", ANNOTATED_CSV)
print("Image directory:", IMAGE_DIR)
print("Saliency directory:", SALIENCY_DIR)

In [ ]:
# CELL 4 — Load and validate annotations

if not ANNOTATED_CSV.exists():
    raise FileNotFoundError(
        f"Annotation file not found: {ANNOTATED_CSV}\n"
        "Set ANNOTATED_CSV to the actual annotated_all_models.csv path."
    )

df = pd.read_csv(ANNOTATED_CSV)

required = ["image_id", "file_name", "human_correct_blip"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required annotation columns: {missing}")

if "error_token_blip" not in df.columns:
    df["error_token_blip"] = pd.NA

df["image_id"] = df["image_id"].astype(str).str.strip()
df["file_name"] = df["file_name"].astype(str).str.strip()
df["human_correct_blip"] = pd.to_numeric(
    df["human_correct_blip"], errors="coerce"
)

df = df.dropna(subset=["human_correct_blip"]).copy()
df["human_correct_blip"] = df["human_correct_blip"].astype(int)

print("Rows:", len(df))
print("Correct:", int((df["human_correct_blip"] == 1).sum()))
print("Incorrect:", int((df["human_correct_blip"] == 0).sum()))
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# CELL 5 — Saliency loading and normalization

def normalize_distribution(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = np.maximum(x, 0.0)

    total = x.sum()
    if total <= eps:
        x = np.ones_like(x, dtype=np.float64)
        total = x.sum()

    return x / total


def resolve_saliency_path(row):
    iid = str(row["image_id"])
    fname = str(row["file_name"])

    candidates = [
        SALIENCY_DIR / f"{Path(fname).stem}.png",
        SALIENCY_DIR / f"{iid}.png",
        SALIENCY_DIR / fname.replace(".jpg", ".png"),
        SALIENCY_DIR / fname.replace(".jpeg", ".png"),
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


def load_saliency(row, spatial_size):
    path = resolve_saliency_path(row)
    if path is None:
        raise FileNotFoundError(
            f"No saliency map found for image_id={row['image_id']} "
            f"file_name={row['file_name']}"
        )

    sal = Image.open(path).convert("L")
    sal = sal.resize((spatial_size, spatial_size), Image.BILINEAR)
    sal = np.asarray(sal, dtype=np.float64)

    return normalize_distribution(sal.flatten())


print("Saliency loader ready.")

In [ ]:
# CELL 6 — Alignment metrics

def js_div(p, q):
    p = normalize_distribution(p)
    q = normalize_distribution(q)
    return float(jensenshannon(p, q, base=None) ** 2)


def kl_div(p, q):
    p = normalize_distribution(p)
    q = normalize_distribution(q)
    eps = 1e-12
    return float(np.sum(p * np.log((p + eps) / (q + eps))))


def cosine_similarity(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)

    denom = np.linalg.norm(p) * np.linalg.norm(q)
    if denom == 0:
        return np.nan

    return float(np.dot(p, q) / denom)


def paired_cohens_d(diffs):
    diffs = np.asarray(diffs, dtype=np.float64)
    if len(diffs) < 2:
        return np.nan

    sd = np.std(diffs, ddof=1)
    return float(np.mean(diffs) / sd) if sd > 0 else np.nan


# Sanity check: identical distributions should have JS≈0 and cosine≈1.
p = np.array([0.1, 0.4, 0.3, 0.2])
q = p.copy()

print("JS:", js_div(p, q))
print("KL:", kl_div(p, q))
print("Cosine:", cosine_similarity(p, q))

In [ ]:
# CELL 7 — Load BLIP

processor = BlipProcessor.from_pretrained(MODEL_ID)

model = BlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

model.eval()

print("Loaded:", MODEL_ID)

In [ ]:
# CELL 8 — Extract real BLIP token-level cross-attention

def extract_blip_attention(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            output_attentions=True,
            return_dict_in_generate=True
        )

    sequence = generated.sequences[0].detach().cpu()
    caption = processor.decode(sequence, skip_special_tokens=True).strip()

    cross_attentions = getattr(generated, "cross_attentions", None)

    if cross_attentions is None or len(cross_attentions) == 0:
        raise RuntimeError(
            "BLIP generation did not return cross_attentions. "
            "Check the installed Transformers version and generation output."
        )

    token_records = []

    for step_idx, step_attention in enumerate(cross_attentions):
        if step_attention is None or len(step_attention) == 0:
            continue

        # Last decoder layer: [batch, heads, query_len, encoder_tokens]
        last_layer = step_attention[-1]

        if last_layer.ndim != 4:
            raise RuntimeError(
                f"Unexpected cross-attention shape at step {step_idx}: "
                f"{tuple(last_layer.shape)}"
            )

        # [heads, query_len, encoder_tokens]
        attn = last_layer[0]

        # Generation normally has query_len=1. Use the final query if not.
        attn = attn[:, -1, :]

        # Mean over attention heads.
        attn = attn.mean(dim=0)

        n_encoder_tokens = int(attn.shape[-1])

        # Infer square patch grid after removing CLS.
        patch_count = n_encoder_tokens - 1
        grid_size = int(round(np.sqrt(patch_count)))

        if grid_size * grid_size != patch_count:
            raise RuntimeError(
                f"Encoder attention has {n_encoder_tokens} tokens; "
                "cannot form a square patch grid after removing CLS."
            )

        spatial = attn[1:].detach().float().cpu().numpy()
        spatial = spatial.reshape(grid_size, grid_size)
        spatial = normalize_distribution(spatial)

        # Generation step i predicts sequence token at i+1 for the
        # standard encoder-decoder generation sequence.
        token_position = step_idx + 1

        token_id = (
            int(sequence[token_position])
            if token_position < len(sequence)
            else None
        )

        token_text = (
            processor.tokenizer.decode(
                [token_id],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=False
            ).strip()
            if token_id is not None
            else ""
        )

        token_records.append({
            "step_idx": step_idx,
            "sequence_position": token_position,
            "token_id": token_id,
            "token_text": token_text,
            "attention": spatial.flatten()
        })

    return {
        "caption": caption,
        "sequence_ids": sequence.numpy(),
        "tokens": token_records,
        "grid_size": grid_size
    }


print("Real-attention extractor defined.")

In [ ]:
# CELL 9 — Smoke test on one image

test_row = df.iloc[0]
test_image = IMAGE_DIR / str(test_row["file_name"])

if not test_image.exists():
    raise FileNotFoundError(f"Test image not found: {test_image}")

test_output = extract_blip_attention(test_image)

print("Caption:", test_output["caption"])
print("Grid:", f"{test_output['grid_size']}x{test_output['grid_size']}")
print("Attention steps:", len(test_output["tokens"]))

print("\nFirst generated tokens:")
for rec in test_output["tokens"][:10]:
    print(
        rec["step_idx"],
        rec["sequence_position"],
        rec["token_id"],
        repr(rec["token_text"]),
        rec["attention"].shape
    )

In [ ]:
# CELL 10 — Map an annotated error token to generated token position

def normalize_token_text(text):
    if text is None or pd.isna(text):
        return ""
    return " ".join(str(text).lower().strip().split())


def find_error_token_position(token_records, error_token):
    """
    Conservative matching.

    Exact decoded-token matching is attempted first. Common tokenizer
    whitespace markers are then stripped. If there is not exactly one
    match, return None rather than guessing.
    """
    target = normalize_token_text(error_token)

    if not target:
        return None

    candidates = []

    for rec in token_records:
        tok = normalize_token_text(rec["token_text"])
        tok_clean = tok.replace("##", "").replace("Ġ", "").replace("▁", "")

        if tok == target or tok_clean == target:
            candidates.append(rec["sequence_position"])

    if len(candidates) == 1:
        return candidates[0]

    return None


print("Token matching helper defined.")

In [ ]:
# CELL 11 — Run real attention extraction and cache it

# First run uses MAX_IMAGES for debugging.
# Set MAX_IMAGES=None for the complete annotated dataset after the pipeline passes.

run_df = df.copy()

if MAX_IMAGES is not None:
    correct_part = run_df[
        run_df["human_correct_blip"] == 1
    ].head(MAX_IMAGES // 2)

    error_part = run_df[
        run_df["human_correct_blip"] == 0
    ].head(MAX_IMAGES - len(correct_part))

    run_df = pd.concat(
        [correct_part, error_part]
    ).drop_duplicates("image_id")

print("Images selected:", len(run_df))

attention_cache = {}
failures = []

for _, row in tqdm(
    run_df.iterrows(),
    total=len(run_df),
    desc="BLIP attention"
):
    iid = str(row["image_id"])
    image_path = IMAGE_DIR / str(row["file_name"])

    if not image_path.exists():
        failures.append({
            "image_id": iid,
            "reason": f"Image missing: {image_path}"
        })
        continue

    try:
        result = extract_blip_attention(image_path)

        attention_cache[iid] = {
            "caption": result["caption"],
            "sequence_ids": result["sequence_ids"].tolist(),
            "grid_size": result["grid_size"],
            "tokens": [
                {
                    "step_idx": x["step_idx"],
                    "sequence_position": x["sequence_position"],
                    "token_id": x["token_id"],
                    "token_text": x["token_text"],
                    "attention": x["attention"].tolist()
                }
                for x in result["tokens"]
            ]
        }

    except Exception as exc:
        failures.append({
            "image_id": iid,
            "reason": repr(exc)
        })

cache_path = CACHE_DIR / "blip_attention_cache.json"

with open(cache_path, "w", encoding="utf-8") as f:
    json.dump(attention_cache, f)

failure_path = CACHE_DIR / "attention_failures.csv"
pd.DataFrame(failures).to_csv(failure_path, index=False)

print("Successful:", len(attention_cache))
print("Failures:", len(failures))
print("Cache:", cache_path)

In [ ]:
# CELL 12 — Compute empirical token-level alignment

rows = []

for _, row in tqdm(
    run_df.iterrows(),
    total=len(run_df),
    desc="Computing alignment"
):
    iid = str(row["image_id"])

    if iid not in attention_cache:
        continue

    cache_item = attention_cache[iid]
    tokens = cache_item["tokens"]
    grid_size = int(cache_item["grid_size"])

    try:
        saliency = load_saliency(row, grid_size)
    except Exception as exc:
        failures.append({
            "image_id": iid,
            "reason": f"Saliency: {exc}"
        })
        continue

    error_position = None

    if int(row["human_correct_blip"]) == 0:
        error_position = find_error_token_position(
            tokens,
            row.get("error_token_blip", None)
        )

    for rec in tokens:
        attn = normalize_distribution(
            np.asarray(rec["attention"], dtype=np.float64)
        )

        rows.append({
            "image_id": iid,
            "sequence_position": rec["sequence_position"],
            "token_id": rec["token_id"],
            "token_text": rec["token_text"],
            "is_error_token": (
                error_position is not None
                and rec["sequence_position"] == error_position
            ),
            "js_divergence": js_div(attn, saliency),
            "kl_divergence": kl_div(attn, saliency),
            "cosine_similarity": cosine_similarity(attn, saliency),
            "attention_map": attn.tolist(),
        })

token_df = pd.DataFrame(rows)

print("Token-level rows:", len(token_df))

if len(token_df):
    print("Images represented:", token_df["image_id"].nunique())
    print("Mapped error tokens:", int(token_df["is_error_token"].sum()))

In [ ]:
# CELL 13 — Caption-level aggregation

if token_df.empty:
    raise RuntimeError("No token-level measurements were produced.")

caption_metrics = (
    token_df.groupby("image_id")
    .agg(
        mean_js=("js_divergence", "mean"),
        mean_kl=("kl_divergence", "mean"),
        mean_cosine=("cosine_similarity", "mean"),
        max_js=("js_divergence", "max"),
        n_tokens=("sequence_position", "count"),
    )
    .reset_index()
)

caption_df = run_df.merge(
    caption_metrics,
    on="image_id",
    how="inner"
)

print(caption_df[
    [
        "image_id",
        "human_correct_blip",
        "mean_js",
        "mean_kl",
        "mean_cosine",
        "max_js",
        "n_tokens"
    ]
].head())

In [ ]:
# CELL 14 — Correct vs incorrect caption-level analysis

correct = caption_df.loc[
    caption_df["human_correct_blip"] == 1,
    "mean_js"
].dropna().to_numpy()

incorrect = caption_df.loc[
    caption_df["human_correct_blip"] == 0,
    "mean_js"
].dropna().to_numpy()

print("Correct n:", len(correct))
print("Incorrect n:", len(incorrect))

if len(correct) >= 2 and len(incorrect) >= 2:
    t_stat, p_value = stats.ttest_ind(
        correct,
        incorrect,
        equal_var=False
    )

    print("Correct mean JS:", correct.mean())
    print("Incorrect mean JS:", incorrect.mean())
    print("Welch t-test:", t_stat, p_value)
else:
    print("Not enough data for caption-level test.")

In [ ]:
# CELL 15 — Paired error-token analysis

error_ids = caption_df.loc[
    caption_df["human_correct_blip"] == 0,
    "image_id"
].astype(str)

error_token_rows = token_df[
    token_df["image_id"].astype(str).isin(error_ids)
].copy()

paired_rows = []

for iid, group in error_token_rows.groupby("image_id"):
    error_rows = group[group["is_error_token"]]
    other_rows = group[~group["is_error_token"]]

    # Require exactly one mapped error token for a clean paired comparison.
    if len(error_rows) != 1 or len(other_rows) == 0:
        continue

    paired_rows.append({
        "image_id": iid,
        "error_token": error_rows.iloc[0]["token_text"],
        "error_js": float(error_rows.iloc[0]["js_divergence"]),
        "other_mean_js": float(other_rows["js_divergence"].mean()),
        "error_kl": float(error_rows.iloc[0]["kl_divergence"]),
        "other_mean_kl": float(other_rows["kl_divergence"].mean()),
        "error_cosine": float(error_rows.iloc[0]["cosine_similarity"]),
        "other_mean_cosine": float(
            other_rows["cosine_similarity"].mean()
        ),
    })

paired_df = pd.DataFrame(paired_rows)

print("Paired error cases:", len(paired_df))

if len(paired_df):
    print(paired_df.head())

In [ ]:
# CELL 16 — Statistical tests, effect size, confidence interval

statistical_results = {
    "n_paired": int(len(paired_df)),
    "paired_tests": {},
}

if len(paired_df) >= 3:
    js_diff = (
        paired_df["error_js"] -
        paired_df["other_mean_js"]
    ).to_numpy()

    kl_diff = (
        paired_df["error_kl"] -
        paired_df["other_mean_kl"]
    ).to_numpy()

    cos_diff = (
        paired_df["error_cosine"] -
        paired_df["other_mean_cosine"]
    ).to_numpy()

    t_js, p_js = ttest_rel(
        paired_df["error_js"],
        paired_df["other_mean_js"]
    )

    t_kl, p_kl = ttest_rel(
        paired_df["error_kl"],
        paired_df["other_mean_kl"]
    )

    t_cos, p_cos = ttest_rel(
        paired_df["error_cosine"],
        paired_df["other_mean_cosine"]
    )

    mw_js, mw_p_js = mannwhitneyu(
        paired_df["error_js"],
        paired_df["other_mean_js"],
        alternative="two-sided"
    )

    d_js = paired_cohens_d(js_diff)

    se_js = np.std(js_diff, ddof=1) / np.sqrt(len(js_diff))
    ci_js = (
        float(np.mean(js_diff) - 1.96 * se_js),
        float(np.mean(js_diff) + 1.96 * se_js),
    )

    statistical_results["paired_tests"] = {
        "js": {"t": float(t_js), "p": float(p_js)},
        "kl": {"t": float(t_kl), "p": float(p_kl)},
        "cosine": {"t": float(t_cos), "p": float(p_cos)},
        "js_mann_whitney_u": float(mw_js),
        "js_mann_whitney_p": float(mw_p_js),
    }

    statistical_results["js_effect"] = {
        "mean_difference_error_minus_other": float(np.mean(js_diff)),
        "cohens_d_paired": float(d_js),
        "ci95": list(ci_js),
    }

    print("=== TOKEN-LEVEL RESULT ===")
    print("Mean JS — error token:", paired_df["error_js"].mean())
    print("Mean JS — other tokens:", paired_df["other_mean_js"].mean())
    print("Difference (error - other):", np.mean(js_diff))
    print("Paired t-test p:", p_js)
    print("Mann-Whitney p:", mw_p_js)
    print("Paired Cohen's d:", d_js)
    print("95% CI:", ci_js)

else:
    print("At least 3 paired cases are required for inferential analysis.")

In [ ]:
# CELL 17 — Empirical visualization

if len(paired_df) >= 1:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].boxplot(
        [
            paired_df["error_js"].dropna(),
            paired_df["other_mean_js"].dropna()
        ],
        labels=["Error token", "Other tokens"]
    )
    axes[0].set_ylabel("JS divergence")
    axes[0].set_title("Token-level visual alignment")

    for _, r in paired_df.iterrows():
        axes[1].plot(
            [0, 1],
            [r["error_js"], r["other_mean_js"]],
            alpha=0.35
        )

    axes[1].scatter(
        np.zeros(len(paired_df)),
        paired_df["error_js"],
        s=20
    )
    axes[1].scatter(
        np.ones(len(paired_df)),
        paired_df["other_mean_js"],
        s=20
    )

    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(["Error", "Other"])
    axes[1].set_ylabel("JS divergence")
    axes[1].set_title("Paired differences")

    plt.tight_layout()
    fig.savefig(
        FIGURES_DIR / "token_level_analysis_clean.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

In [ ]:
# CELL 18 — Visualize REAL attention for one mapped error case

if len(paired_df) > 0:
    example_id = str(paired_df.iloc[0]["image_id"])

    cache_item = attention_cache[example_id]

    row = run_df.loc[
        run_df["image_id"].astype(str) == example_id
    ].iloc[0]

    grid = int(cache_item["grid_size"])
    saliency = load_saliency(row, grid).reshape(grid, grid)

    group = token_df[
        token_df["image_id"].astype(str) == example_id
    ].copy()

    error_row = group[group["is_error_token"]]

    if len(error_row) == 1:
        error_map = np.asarray(
            error_row.iloc[0]["attention_map"]
        ).reshape(grid, grid)

        other_row = group[~group["is_error_token"]].iloc[0]
        other_map = np.asarray(
            other_row["attention_map"]
        ).reshape(grid, grid)

        fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

        axes[0].imshow(saliency, cmap="hot")
        axes[0].set_title("Human saliency")
        axes[0].axis("off")

        axes[1].imshow(error_map, cmap="hot")
        axes[1].set_title(
            f"Error token: {error_row.iloc[0]['token_text']}"
        )
        axes[1].axis("off")

        axes[2].imshow(other_map, cmap="hot")
        axes[2].set_title(
            f"Other token: {other_row['token_text']}"
        )
        axes[2].axis("off")

        plt.tight_layout()
        fig.savefig(
            FIGURES_DIR / "real_attention_example.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()
    else:
        print("Could not create example visualization.")

In [ ]:
# CELL 19 — Save clean outputs

token_output = OUTPUT_DIR / "token_attention_metrics.csv"
caption_output = OUTPUT_DIR / "caption_attention_metrics.csv"
paired_output = OUTPUT_DIR / "paired_error_token_analysis.csv"
stats_output = OUTPUT_DIR / "statistical_results_clean.json"

token_export = token_df.drop(
    columns=["attention_map"],
    errors="ignore"
)

token_export.to_csv(token_output, index=False)
caption_df.to_csv(caption_output, index=False)
paired_df.to_csv(paired_output, index=False)

with open(stats_output, "w", encoding="utf-8") as f:
    json.dump(statistical_results, f, indent=2)

print("Saved:")
print(token_output)
print(caption_output)
print(paired_output)
print(stats_output)